<a href="https://colab.research.google.com/github/parulgoell/ai-attendance-agent/blob/main/Emergency_Response_Dispatchers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Emergency Response Dispatchers — corrected single-cell code with line-by-line comments

import numpy as np                # For numeric operations and distance calculations

np.random.seed(0)                 # Fix randomness so results are reproducible

num_units = 8                     # Number of emergency units (ambulances / fire trucks)
num_incidents = 12                # Number of incidents to dispatch to

# Random (x, y) integer coordinates in a 100x100 city grid for emergency units
units = np.random.randint(0, 100, (num_units, 2))

# Random (x, y) integer coordinates in the same grid for incidents
incidents = np.random.randint(0, 100, (num_incidents, 2))

# Choice of assignment behavior:
# If reuse_units=True -> units can respond to multiple incidents (simple, common test model)
# If reuse_units=False -> units are used at most once; if incidents > units, leftover incidents are assigned
#                      to the nearest unit anyway (fallback) to avoid 'inf' distances.
reuse_units = True

# Keep track of which units have already been used (only meaningful when reuse_units=False)
used_units = set()

# Store computed response times (distance = time, for this simple model)
response_times = []

# For each incident, find the nearest unit according to the chosen policy
for inc in incidents:                        # Iterate every incident coordinate
    best_unit = None                         # Index of the best unit candidate (initially none)
    best_dist = float("inf")                 # Distance to the best unit (start very large)

    for idx, unit in enumerate(units):       # Check each unit's distance to the incident
        if not reuse_units and idx in used_units:
            # If we're forbidding reuse and this unit is already used, skip it for now
            continue

        # Euclidean distance between unit and incident
        dist = np.linalg.norm(unit - inc)

        # Update best unit if this one is closer
        if dist < best_dist:
            best_dist = dist
            best_unit = idx

    # Safeguard: if no unit was selectable because reuse is False and all units used,
    # fall back to allowing reuse by assigning the globally nearest unit (prevents inf).
    if best_unit is None:
        # Find nearest unit regardless of used status (fallback)
        for idx, unit in enumerate(units):
            dist = np.linalg.norm(unit - inc)
            if dist < best_dist:
                best_dist = dist
                best_unit = idx
        # Note: we do NOT change used_units here because this is a fallback assignment.

    # If we are using one-shot assignments, mark the selected unit as used
    if not reuse_units:
        used_units.add(best_unit)

    # Append the chosen response time (distance) to the list
    response_times.append(best_dist)

# Compute the average response time across incidents
average_response = np.mean(response_times)

# Print a friendly summary of results and some diagnostics
print("Average Response Time:", average_response)   # Average of all recorded response times
print("Number of units:", num_units)                # How many units were simulated
print("Number of incidents:", num_incidents)        # How many incidents were simulated
print("Reuse units?:", reuse_units)                 # Which dispatch policy was used


Average Response Time: 17.04103704643298
Number of units: 8
Number of incidents: 12
Reuse units?: True
